# 1. Descarga y organización del corpus

In [ ]:
from pathlib import Path  # Clase Path: maneja rutas de archivos y carpetas sin depender del sistema operativo
import urllib.request  # Módulo para descargar archivos desde una URL
import zipfile  # Módulo para leer y descomprimir archivos .zip

# URL del corpus Mini Speech Commands (subconjunto del dataset Speech Commands de TensorFlow)
url = "https://storage.googleapis.com/download.tensorflow.org/data/mini_speech_commands.zip"

carpeta_datos = Path.cwd() / "Datos"  # Ruta de la carpeta "Datos" dentro del directorio de trabajo actual
carpeta_datos.mkdir(exist_ok=True)  # Crea la carpeta; si ya existe, no marca error

archivo_zip = carpeta_datos / "mini_speech_commands.zip"  # Ruta donde se guardará el ZIP descargado
carpeta_corpus = carpeta_datos / "mini_speech_commands"  # Ruta de la carpeta con el corpus ya descomprimido

if not archivo_zip.exists():  # Si el ZIP todavía no se ha descargado...
    print("Descargando Mini Speech Commands")  # ...avisa que empieza la descarga
    urllib.request.urlretrieve(url, archivo_zip)  # Descarga el archivo de la URL y lo guarda en archivo_zip
    print("Descarga completada")  # Confirma que terminó la descarga
else:  # Si el ZIP ya existe...
    print("El archivo ZIP ya existe")  # ...avisa que no hace falta descargarlo otra vez

if not carpeta_corpus.exists():  # Si el corpus todavía no está descomprimido...
    print("Descomprimiendo archivos")  # ...avisa que empieza la descompresión
    with zipfile.ZipFile(archivo_zip, "r") as zip_ref:  # Abre el ZIP en modo lectura (se cierra solo al salir del with)
        zip_ref.extractall(carpeta_datos)  # Extrae todo el contenido dentro de la carpeta Datos
    print("Descompresión completada")  # Confirma que terminó la descompresión
else:  # Si el corpus ya existe...
    print("El corpus ya está descomprimido")  # ...avisa que no hace falta descomprimir

comandos = sorted([  # Lista ordenada alfabéticamente con los nombres de los comandos
    carpeta.name  # Nombre de cada subcarpeta (cada carpeta es un comando de voz)
    for carpeta in carpeta_corpus.iterdir()  # Recorre todo el contenido de la carpeta del corpus
    if carpeta.is_dir()])  # Se queda solo con las carpetas, ignorando archivos sueltos

print("\nCOMANDOS DISPONIBLES")  # Título del listado (el \n deja una línea en blanco antes)
print("-" * 40)  # Línea separadora de 40 guiones

total_grabaciones = 0  # Contador acumulado de grabaciones de todos los comandos

for comando in comandos:  # Recorre cada comando
    archivos = list((carpeta_corpus / comando).glob("*.wav"))  # Lista todos los archivos .wav de la carpeta del comando
    cantidad = len(archivos)  # Cuenta cuántas grabaciones tiene ese comando
    total_grabaciones += cantidad  # Suma esa cantidad al total general

    print(f"{comando:10s}: {cantidad} grabaciones")  # Imprime el comando alineado a 10 caracteres y su cantidad

print("-" * 40)  # Línea separadora final
print(f"Total de grabaciones: {total_grabaciones}")  # Muestra el total de grabaciones del corpus
print(f"\nUbicación del corpus:\n{carpeta_corpus}")  # Muestra la ruta donde quedó el corpus

# 2. Exploración y selección de una señal de voz

In [ ]:
import numpy as np  # Librería para cálculo numérico y manejo de arreglos
import librosa  # Librería para análisis y carga de audio
from IPython.display import Audio, display  # Herramientas para reproducir audio dentro del notebook

comando_seleccionado = "yes"  # Comando que se va a analizar

archivos_audio = sorted(  # Lista ordenada de archivos de audio del comando
    (carpeta_corpus / comando_seleccionado).glob("*.wav"))  # Busca todos los .wav de la carpeta del comando

archivo_seleccionado = archivos_audio[0]  # Elige la primera grabación de la lista

senal, fs = librosa.load(  # Carga el audio: devuelve la señal (arreglo) y la frecuencia de muestreo
    archivo_seleccionado,  # Ruta del archivo a cargar
    sr=None,  # sr=None conserva la frecuencia de muestreo original (no remuestrea)
    mono=True)  # Convierte a un solo canal (mono)

numero_muestras = len(senal)  # Cantidad total de muestras de la señal
duracion = numero_muestras / fs  # Duración en segundos = muestras / frecuencia de muestreo
amplitud_minima = np.min(senal)  # Valor más bajo de amplitud en la señal
amplitud_maxima = np.max(senal)  # Valor más alto de amplitud en la señal

print("CARACTERÍSTICAS DE LA GRABACIÓN")  # Título del resumen
print("-" * 45)  # Línea separadora de 45 guiones
print(f"Comando: {comando_seleccionado}")  # Muestra el comando elegido
print(f"Archivo: {archivo_seleccionado.name}")  # Muestra el nombre del archivo
print(f"Frecuencia de muestreo: {fs} Hz")  # Muestra la frecuencia de muestreo
print(f"Número de muestras: {numero_muestras}")  # Muestra el total de muestras
print(f"Duración: {duracion:.4f} segundos")  # Muestra la duración con 4 decimales
print(f"Amplitud mínima: {amplitud_minima:.4f}")  # Muestra la amplitud mínima con 4 decimales
print(f"Amplitud máxima: {amplitud_maxima:.4f}")  # Muestra la amplitud máxima con 4 decimales
print("-" * 45)  # Línea separadora final

print("\nREPRODUCCIÓN DE LA GRABACIÓN ORIGINAL")  # Título de la sección de reproducción
display(Audio(senal, rate=fs))  # Muestra un reproductor de audio con la señal y su frecuencia de muestreo

# 3. Visualización de la señal de voz

In [ ]:
import matplotlib.pyplot as plt  # Librería para graficar

tiempo = np.arange(len(senal)) / fs  # Eje del tiempo: índices 0..N-1 divididos entre fs dan segundos

plt.figure(figsize=(12, 5))  # Crea una figura de 12x5 pulgadas

plt.plot(tiempo, senal, color="blue", linewidth=0.7)  # Grafica la señal (amplitud vs. tiempo) en azul con línea delgada

plt.title(  # Título de la gráfica
    f"Señal de voz original - Comando: {comando_seleccionado}",  # Texto con el comando seleccionado
    fontsize=14)  # Tamaño de letra del título

plt.xlabel("Tiempo (segundos)", fontsize=12)  # Etiqueta del eje X
plt.ylabel("Amplitud", fontsize=12)  # Etiqueta del eje Y

plt.xlim(0, duracion)  # Eje X va de 0 a la duración total de la señal
plt.ylim(-1, 1)  # Eje Y fijo entre -1 y 1 (rango normalizado de librosa)

plt.grid(True, alpha=0.3)  # Activa la cuadrícula con transparencia del 30 %

plt.tight_layout()  # Ajusta los márgenes para que nada se corte
plt.show()  # Muestra la gráfica

# 4. Visualización de las muestras individuales

In [ ]:
tiempo_inicio = 0.48  # Inicio del fragmento a analizar (en segundos)
tiempo_fin = 0.485  # Fin del fragmento a analizar (en segundos)

indice_inicio = int(tiempo_inicio * fs)  # Convierte el tiempo inicial a índice de muestra (tiempo * fs)
indice_fin = int(tiempo_fin * fs)  # Convierte el tiempo final a índice de muestra

fragmento = senal[indice_inicio:indice_fin]  # Recorta la señal entre los dos índices
tiempo_fragmento = tiempo[indice_inicio:indice_fin]  # Recorta el eje del tiempo con los mismos índices

print("ANÁLISIS DEL FRAGMENTO")  # Título del resumen
print("-" * 40)  # Línea separadora de 40 guiones
print(f"Tiempo inicial: {tiempo_inicio} s")  # Muestra el tiempo donde empieza el fragmento
print(f"Tiempo final: {tiempo_fin} s")  # Muestra el tiempo donde termina el fragmento
print(f"Duración: {(tiempo_fin - tiempo_inicio) * 1000:.2f} ms")  # Duración del fragmento en milisegundos
print(f"Número de muestras: {len(fragmento)}")  # Cantidad de muestras dentro del fragmento

plt.figure(figsize=(12, 5))  # Crea una figura de 12x5 pulgadas

plt.plot(  # Grafica la señal como línea que une las muestras
    tiempo_fragmento,  # Eje X: tiempo del fragmento
    fragmento,  # Eje Y: amplitud del fragmento
    color="blue",  # Línea azul
    linewidth=1,  # Grosor de la línea
    label="Muestras conectadas")  # Nombre que aparece en la leyenda

plt.stem(  # Grafica cada muestra como una línea vertical con un punto (señal discreta)
    tiempo_fragmento,  # Eje X: tiempo del fragmento
    fragmento,  # Eje Y: amplitud del fragmento
    linefmt="gray",  # Líneas verticales en gris
    markerfmt="ro",  # Marcadores: círculos rojos
    basefmt=" ",  # Oculta la línea base (eje horizontal del stem)
    label="Muestras discretas")  # Nombre que aparece en la leyenda

plt.title("Visualización de muestras individuales", fontsize=14)  # Título de la gráfica
plt.xlabel("Tiempo (segundos)", fontsize=12)  # Etiqueta del eje X
plt.ylabel("Amplitud", fontsize=12)  # Etiqueta del eje Y

plt.grid(True, alpha=0.3)  # Cuadrícula con transparencia del 30 %
plt.legend()  # Muestra la leyenda con los nombres de cada serie
plt.tight_layout()  # Ajusta los márgenes de la figura
plt.show()  # Muestra la gráfica

# 5. Comparación de frecuencias de muestreo

In [ ]:
from scipy.signal import resample_poly  # Función de remuestreo con filtro antialiasing incluido

frecuencias = [16000, 8000, 4000, 2000]  # Frecuencias de muestreo (Hz) que se van a comparar

senales_muestreadas = {}  # Diccionario: frecuencia -> señal remuestreada

for frecuencia in frecuencias:  # Recorre cada frecuencia objetivo

    if frecuencia == fs:  # Si la frecuencia objetivo es la original...
        senal_nueva = senal.copy()  # ...solo copia la señal, sin remuestrear
    else:  # Si es distinta...
        senal_nueva = resample_poly(  # ...remuestrea la señal
            senal,  # Señal original
            up = frecuencia,  # Factor de interpolación (frecuencia nueva)
            down = fs)  # Factor de decimación (frecuencia original)

    senales_muestreadas[frecuencia] = senal_nueva  # Guarda la señal remuestreada en el diccionario

fig, ejes = plt.subplots(4, 1, figsize = (12, 12))  # Figura con 4 gráficas apiladas (una por frecuencia)

for i, frecuencia in enumerate(frecuencias):  # Recorre las frecuencias con su índice (i = número de gráfica)

    senal_actual = senales_muestreadas[frecuencia]  # Señal remuestreada de esta frecuencia

    tiempo_actual = np.arange(len(senal_actual)) / frecuencia  # Eje de tiempo con la frecuencia correspondiente

    mascara = (  # Máscara booleana que selecciona solo el fragmento de interés
        (tiempo_actual >= tiempo_inicio) &  # Tiempos mayores o iguales al inicio
        (tiempo_actual < tiempo_fin))  # Y menores al fin

    t = tiempo_actual[mascara]  # Tiempos dentro del fragmento
    x = senal_actual[mascara]  # Amplitudes dentro del fragmento

    ejes[i].stem(  # Dibuja cada muestra como una línea vertical con punto
        t,  # Eje X: tiempos
        x,  # Eje Y: amplitudes
        linefmt = "gray",  # Líneas verticales grises
        markerfmt = "ro",  # Puntos rojos
        basefmt = " ")  # Sin línea base

    ejes[i].plot(t, x, color = "blue", linewidth = 1)  # Une las muestras con una línea azul

    ejes[i].set_title(  # Título de esta gráfica
        f"Frecuencia: {frecuencia} Hz - {len(x)} muestras")  # Frecuencia y cantidad de muestras del fragmento

    ejes[i].set_ylabel("Amplitud")  # Etiqueta del eje Y
    ejes[i].set_xlim(tiempo_inicio, tiempo_fin)  # Límites del eje X: el fragmento analizado
    ejes[i].set_ylim(-0.25, 0.25)  # Límites del eje Y fijos para poder comparar
    ejes[i].grid(True, alpha = 0.3)  # Cuadrícula tenue

ejes[-1].set_xlabel("Tiempo (segundos)")  # Etiqueta de X solo en la última gráfica

plt.suptitle(  # Título general de la figura
    "Comparación de diferentes frecuencias de muestreo",  # Texto del título
    fontsize = 15)  # Tamaño de letra

plt.tight_layout()  # Ajusta márgenes para que nada se encime
plt.show()  # Muestra la figura

print("\nCOMPARACIÓN DE FRECUENCIAS DE MUESTREO")  # Título de la tabla de resultados
print("-" * 55)  # Línea separadora

for frecuencia in frecuencias:  # Recorre cada frecuencia
    cantidad = len(senales_muestreadas[frecuencia])  # Número total de muestras de esa señal

    print(f"{frecuencia:5d} Hz | "  # Frecuencia alineada a 5 dígitos
          f"{cantidad:5d} muestras | "  # Cantidad de muestras alineada a 5 dígitos
          f"{cantidad/frecuencia:.3f} segundos")  # Duración = muestras / frecuencia (3 decimales)

# 6. Reproducción de señales muestreadas

In [ ]:
fs_reproduccion = 16000  # Frecuencia a la que se reproducirán todas las señales

for frecuencia in frecuencias:  # Recorre cada frecuencia de muestreo

    senal_actual = senales_muestreadas[frecuencia]  # Señal remuestreada correspondiente

    duracion_actual = len(senal_actual) / frecuencia  # Duración en segundos
    amplitud_maxima = np.max(np.abs(senal_actual))  # Valor absoluto máximo de la señal
    rms = np.sqrt(np.mean(senal_actual**2))  # Valor RMS (raíz de la media de los cuadrados)

    print("=" * 50)  # Línea separadora
    print(f"FRECUENCIA DE MUESTREO: {frecuencia} Hz")  # Frecuencia actual
    print(f"Número de muestras: {len(senal_actual)}")  # Total de muestras
    print(f"Duración: {duracion_actual:.3f} segundos")  # Duración con 3 decimales
    print(f"Amplitud máxima absoluta: {amplitud_maxima:.4f}")  # Amplitud máxima con 4 decimales
    print(f"Amplitud RMS: {rms:.4f}")  # RMS con 4 decimales

    if frecuencia != fs_reproduccion:  # Si la señal no está a 16000 Hz...

        senal_reproduccion = resample_poly(  # ...la lleva a 16000 Hz para reproducirla
            senal_actual,  # Señal a convertir
            up = fs_reproduccion,  # Frecuencia destino
            down = frecuencia)  # Frecuencia actual

    else:  # Si ya está a 16000 Hz...
        senal_reproduccion = senal_actual.copy()  # ...solo la copia

    display(Audio(senal_reproduccion,  # Muestra el reproductor con la señal
                  rate = fs_reproduccion,  # Frecuencia de reproducción
                  normalize = False))  # No normaliza el volumen (respeta la amplitud real)

# 7. Comparación del muestreo con y sin filtro

In [ ]:
fs_nueva = 2000  # Nueva frecuencia de muestreo (Hz)

# METODO 1. REMUESTREO CON FILTRO ANTIALIASING
senal_con_filtro = resample_poly(senal, up = fs_nueva, down = fs)  # Remuestrea aplicando filtro pasa-bajas antes de reducir

# METODO 2. REDUCCIÓN SIN FILTRO ANTIALIASING
factor = fs // fs_nueva  # Factor de reducción (división entera): 16000 // 2000 = 8

senal_sin_filtro = senal[::factor]  # Toma una de cada "factor" muestras, sin filtrar

print("COMPARACIÓN DE MÉTODOS DE MUESTREO")  # Título del resumen
print("-" * 55)  # Línea separadora

print(f"Frecuencia original: {fs} Hz")  # Frecuencia original
print(f"Nueva frecuencia: {fs_nueva} Hz")  # Frecuencia nueva
print(f"Factor de reducción: {factor}")  # Factor de reducción

print("\nCON FILTRO ANTIALIASING")  # Sección del método 1
print(f"Número de muestras: {len(senal_con_filtro)}")  # Muestras de la señal filtrada
print(f"Amplitud RMS: {np.sqrt(np.mean(senal_con_filtro**2)):.6f}")  # RMS con 6 decimales

print("\nSIN FILTRO ANTIALIASING")  # Sección del método 2
print(f"Número de muestras: {len(senal_sin_filtro)}")  # Muestras de la señal sin filtrar
print(f"Amplitud RMS: {np.sqrt(np.mean(senal_sin_filtro**2)):.6f}")  # RMS con 6 decimales

tiempo_con_filtro = np.arange(len(senal_con_filtro)) / fs_nueva  # Eje de tiempo de la señal con filtro
tiempo_sin_filtro = np.arange(len(senal_sin_filtro)) / fs_nueva  # Eje de tiempo de la señal sin filtro

fig, ejes = plt.subplots(2, 1, figsize = (12, 8))  # Figura con 2 gráficas apiladas

ejes[0].plot(tiempo_con_filtro,  # Eje X: tiempo
             senal_con_filtro,  # Eje Y: señal con filtro
             color = "blue")  # Línea azul

ejes[0].set_title("Muestreo con filtro antialiasing")  # Título de la primera gráfica
ejes[0].set_ylabel("Amplitud")  # Etiqueta del eje Y
ejes[0].grid(True, alpha = 0.3)  # Cuadrícula tenue

ejes[1].plot(tiempo_sin_filtro,  # Eje X: tiempo
             senal_sin_filtro,  # Eje Y: señal sin filtro
             color = "red")  # Línea roja

ejes[1].set_title("Muestreo sin filtro antialiasing")  # Título de la segunda gráfica
ejes[1].set_xlabel("Tiempo (segundos)")  # Etiqueta del eje X
ejes[1].set_ylabel("Amplitud")  # Etiqueta del eje Y
ejes[1].grid(True, alpha = 0.3)  # Cuadrícula tenue

limite = max(np.max(np.abs(senal_con_filtro)),  # Amplitud máxima de la señal con filtro
             np.max(np.abs(senal_sin_filtro))  # Amplitud máxima de la señal sin filtro
            ) * 1.1  # Toma la mayor y agrega 10 % de margen

for eje in ejes:  # Aplica lo mismo a ambas gráficas
    eje.set_xlim(0, 1)  # Eje X de 0 a 1 segundo
    eje.set_ylim(-limite, limite)  # Mismo rango en Y para poder compararlas

plt.tight_layout()  # Ajusta márgenes
plt.show()  # Muestra la figura

audio_con_filtro = resample_poly(senal_con_filtro,  # Regresa la señal con filtro...
                                 up = fs,  # ...a la frecuencia original para reproducirla
                                 down = fs_nueva)  # Desde la frecuencia reducida
audio_sin_filtro = resample_poly(senal_sin_filtro,  # Regresa la señal sin filtro...
                                 up = fs,  # ...a la frecuencia original para reproducirla
                                 down = fs_nueva)  # Desde la frecuencia reducida

print("\nREPRODUCCIÓN CON FILTRO ANTIALIASING")  # Título de la reproducción 1
display(Audio(audio_con_filtro, rate = fs, normalize = False))  # Reproduce el audio con filtro

print("\nREPRODUCCIÓN SIN FILTRO ANTIALIASING")  # Título de la reproducción 2
display(Audio(audio_sin_filtro, rate = fs, normalize = False))  # Reproduce el audio sin filtro

# 8. Demostración del aliasing con una señal senoidal

In [ ]:
fs_original = 16000  # Frecuencia de muestreo original (Hz)
fs_reducida = 2000  # Frecuencia de muestreo reducida (Hz); su Nyquist es 1000 Hz
frecuencia_senoidal = 3500  # Frecuencia de la senoide (Hz), mayor que el Nyquist de fs_reducida
duracion_senoidal = 1.0  # Duración de la señal (segundos)

t_original = np.arange(  # Eje de tiempo de la senoide
    int(fs_original * duracion_senoidal)) / fs_original  # Muestras totales / frecuencia de muestreo

senal_senoidal = 0.7 * np.sin(  # Senoide de amplitud 0.7
    2 * np.pi * frecuencia_senoidal * t_original)  # Argumento: 2·pi·f·t

senoidal_filtrada = resample_poly(senal_senoidal,  # Reduce a 2000 Hz CON filtro antialiasing
                                  up = fs_reducida,  # Frecuencia destino
                                  down = fs_original)  # Frecuencia origen

factor_reduccion = fs_original // fs_reducida  # Factor de reducción: 16000 // 2000 = 8

senoidal_sin_filtro = senal_senoidal[::factor_reduccion]  # Reduce a 2000 Hz SIN filtro (una de cada 8 muestras)

def calcular_espectro(x, frecuencia_muestreo):  # Función que calcula el espectro de amplitud de una señal

    ventana = np.hanning(len(x))  # Ventana de Hanning para reducir fugas espectrales
    x_ventana = x * ventana  # Aplica la ventana a la señal

    espectro = np.fft.rfft(x_ventana)  # FFT para señales reales (solo frecuencias positivas)

    frecuencias_fft = np.fft.rfftfreq(len(x),  # Eje de frecuencias que corresponde a la FFT
                                      d = 1/frecuencia_muestreo)  # d = periodo de muestreo

    amplitudes = (2 * np.abs(espectro) / np.sum(ventana))  # Amplitud normalizada (compensa la ventana)

    return frecuencias_fft, amplitudes  # Devuelve frecuencias y amplitudes

f_original, a_original = calcular_espectro(  # Espectro de la señal original
    senal_senoidal, fs_original)  # Señal y su frecuencia de muestreo

f_filtrada, a_filtrada = calcular_espectro(  # Espectro de la señal reducida con filtro
    senoidal_filtrada, fs_reducida)  # Señal y su frecuencia de muestreo

f_sin_filtro, a_sin_filtro = calcular_espectro(  # Espectro de la señal reducida sin filtro
    senoidal_sin_filtro, fs_reducida)  # Señal y su frecuencia de muestreo

fig, ejes = plt.subplots(3, 1, figsize = (12, 10))  # Figura con 3 gráficas apiladas

ejes[0].plot(f_original, a_original, color = "blue")  # Espectro original en azul
ejes[0].set_title("Señal original - 3500 Hz")  # Título de la gráfica 1
ejes[0].set_xlim(0, 8000)  # Eje X hasta 8000 Hz (Nyquist de 16000 Hz)

ejes[1].plot(f_filtrada, a_filtrada, color = "green")  # Espectro con filtro en verde
ejes[1].set_title("Remuestreo a 2000 Hz CON filtro antialiasing")  # Título de la gráfica 2
ejes[1].set_xlim(0, 1000)  # Eje X hasta 1000 Hz (Nyquist de 2000 Hz)

ejes[2].plot(f_sin_filtro, a_sin_filtro, color = "red")  # Espectro sin filtro en rojo
ejes[2].set_title("Remuestreo a 2000 Hz SIN filtro antialiasing")  # Título de la gráfica 3
ejes[2].set_xlim(0, 1000)  # Eje X hasta 1000 Hz

for eje in ejes:  # Configuración común para las 3 gráficas
    eje.set_ylabel("Amplitud espectral")  # Etiqueta del eje Y
    eje.set_xlabel("Frecuencia (Hz)")  # Etiqueta del eje X
    eje.set_ylim(0, 0.8)  # Mismo rango en Y para comparar
    eje.grid(True, alpha = 0.3)  # Cuadrícula tenue

plt.suptitle("Demostración del fenómeno de aliasing",  # Título general de la figura
             fontsize = 15)  # Tamaño de letra

plt.tight_layout()  # Ajusta márgenes
plt.show()  # Muestra la figura

frecuencia_alias = abs(  # Calcula la frecuencia aparente (alias) en valor absoluto
    frecuencia_senoidal -  # Frecuencia real menos...
    round(frecuencia_senoidal/fs_reducida) * fs_reducida)  # ...el múltiplo de fs_reducida más cercano

print("DEMOSTRACIÓN DEL ALIASING")  # Título del resumen
print("-" * 45)  # Línea separadora
print(f"Frecuencia original: {frecuencia_senoidal} Hz")  # Frecuencia real de la senoide
print(f"Frecuencia de muestreo original: {fs_original} Hz")  # fs original
print(f"Frecuencia de muestreo reducida: {fs_reducida} Hz")  # fs reducida
print(f"Frecuencia de Nyquist: {fs_reducida/2} Hz")  # Límite máximo representable con fs_reducida
print(f"Frecuencia aparente esperada: {frecuencia_alias} Hz")  # Alias esperado (aquí: |3500 - 4000| = 500 Hz)

# 9. Cuantificación de la señal de voz

In [ ]:
resoluciones = [4, 8, 16]  # Resoluciones de cuantificación en bits

senales_cuantificadas = {}  # Diccionario: bits -> señal cuantificada

def cuantificar_senal(x, bits):  # Función de cuantificación uniforme de una señal en [-1, 1)

    escala = 2 ** (bits - 1)  # Cantidad de niveles por cada lado del cero

    niveles = np.rint(x * escala)  # Escala la señal y redondea al entero más cercano

    niveles = np.clip(niveles,  # Limita los valores al rango permitido
                      -escala,  # Mínimo: -2^(bits-1)
                      escala - 1)  # Máximo: 2^(bits-1) - 1

    senal_reconstruida = niveles / escala  # Regresa los enteros al rango original [-1, 1)

    return senal_reconstruida  # Devuelve la señal cuantificada

for bits in resoluciones:  # Recorre cada resolución

    senal_cuantificada = cuantificar_senal(senal, bits)  # Cuantifica la señal con esa cantidad de bits

    senales_cuantificadas[bits] = senal_cuantificada  # La guarda en el diccionario

print("CUANTIFICACIÓN DE LA SEÑAL DE VOZ")  # Título del resumen
print("-" * 65)  # Línea separadora

for bits in resoluciones:  # Recorre cada resolución

    niveles = 2 ** bits  # Total de niveles posibles

    tasa_bits = fs * bits / 1000  # Tasa de bits en kbps = fs · bits / 1000

    print(f"\nRESOLUCIÓN: {bits} bits")  # Resolución actual
    print(f"Niveles de cuantificación posibles: {niveles}")  # Niveles posibles
    print(f"Frecuencia de muestreo: {fs} Hz")  # Frecuencia de muestreo
    print(f"Tasa de bits PCM mono: {tasa_bits:.0f} kbps")  # Tasa de bits sin decimales

fig, ejes = plt.subplots(3, 1, figsize = (12, 11))  # Figura con 3 gráficas apiladas

for i, bits in enumerate(resoluciones):  # Recorre las resoluciones con su índice

    senal_actual = senales_cuantificadas[bits]  # Señal cuantificada con esos bits

    fragmento_cuantificado = senal_actual[  # Recorta la señal cuantificada
        indice_inicio:indice_fin]  # Mismo fragmento que se analizó en la celda 4

    ejes[i].plot(tiempo_fragmento,  # Eje X: tiempo del fragmento
                 fragmento,  # Eje Y: señal original
                 color = "blue",  # Línea azul
                 linewidth = 1,  # Grosor de la línea
                 label = "Señal original")  # Texto de la leyenda

    ejes[i].step(tiempo_fragmento,  # Gráfica en escalones: así se ve la cuantificación
                 fragmento_cuantificado,  # Eje Y: señal cuantificada
                 where = "mid",  # Los escalones se centran en cada muestra
                 color = "red",  # Línea roja
                 linewidth = 1.2,  # Grosor de la línea
                 label = f"Cuantificación de {bits} bits")  # Texto de la leyenda

    ejes[i].set_title(  # Título de la gráfica
        f"Cuantificación uniforme de {bits} bits")  # Indica la resolución

    ejes[i].set_ylabel("Amplitud")  # Etiqueta del eje Y
    ejes[i].set_xlim(tiempo_inicio, tiempo_fin)  # Límites de X: el fragmento
    ejes[i].set_ylim(-0.25, 0.25)  # Límites de Y fijos
    ejes[i].grid(True, alpha = 0.3)  # Cuadrícula tenue
    ejes[i].legend()  # Muestra la leyenda

ejes[-1].set_xlabel("Tiempo (segundos)")  # Etiqueta de X en la última gráfica

plt.suptitle(  # Título general
    "Comparación de diferentes resoluciones de cuantificación",  # Texto del título
    fontsize = 15)  # Tamaño de letra

plt.tight_layout()  # Ajusta márgenes
plt.show()  # Muestra la figura

# 10. Evaluación del error de cuantificación

In [ ]:
resultados_mse = {}  # Diccionario: bits -> error cuadrático medio (MSE)
resultados_rmse = {}  # Diccionario: bits -> raíz del MSE (RMSE)
resultados_sqnr = {}  # Diccionario: bits -> relación señal a ruido de cuantificación (SQNR)

senal_original = senal.astype(np.float64)  # Convierte la señal a float64 para mayor precisión

potencia_senal = np.mean(senal_original**2)  # Potencia promedio de la señal original

print("EVALUACIÓN DEL ERROR DE CUANTIFICACIÓN")  # Título de la tabla
print("-" * 75)  # Línea separadora

print(f"{'Bits':<8}"  # Encabezado "Bits" alineado a la izquierda en 8 caracteres
      f"{'MSE':<18}"  # Encabezado "MSE" en 18 caracteres
      f"{'RMSE':<18}"  # Encabezado "RMSE" en 18 caracteres
      f"{'SQNR (dB)':<15}")  # Encabezado "SQNR (dB)" en 15 caracteres
print("-" * 75)  # Línea separadora

for bits in resoluciones:  # Recorre cada resolución

    senal_actual = senales_cuantificadas[bits].astype(np.float64)  # Señal cuantificada en float64

    error = senal_original - senal_actual  # Error de cuantificación: original menos cuantificada

    mse = np.mean(error**2)  # Error cuadrático medio

    rmse = np.sqrt(mse)  # Raíz del error cuadrático medio

    if mse == 0:  # Si no hay error...
        sqnr = np.inf  # ...la SQNR es infinita (evita dividir entre cero)
    else:  # Si hay error...
        sqnr = 10 * np.log10(potencia_senal / mse)  # SQNR en dB = 10·log10(potencia señal / MSE)

    resultados_mse[bits] = mse  # Guarda el MSE
    resultados_rmse[bits] = rmse  # Guarda el RMSE
    resultados_sqnr[bits] = sqnr  # Guarda la SQNR

    print(f"{bits:<8}"  # Bits alineado a la izquierda en 8 caracteres
          f"{mse:<18.8e}"  # MSE en notación científica con 8 decimales
          f"{rmse:<18.8e}"  # RMSE en notación científica con 8 decimales
          f"{sqnr:<15.2f}")  # SQNR con 2 decimales

fig, ejes = plt.subplots(3, 1, figsize = (12, 10))  # Figura con 3 gráficas apiladas

for i, bits in enumerate(resoluciones):  # Recorre las resoluciones con su índice

    error = (senal_original -  # Error = señal original...
             senales_cuantificadas[bits])  # ...menos la señal cuantificada

    error_fragmento = error[indice_inicio:indice_fin]  # Recorta el error al fragmento analizado

    ejes[i].plot(tiempo_fragmento,  # Eje X: tiempo del fragmento
                 error_fragmento,  # Eje Y: error en el fragmento
                 color = "red",  # Línea roja
                 linewidth = 1)  # Grosor de la línea

    ejes[i].set_title(  # Título de la gráfica
        f"Error de cuantificación - {bits} bits")  # Indica la resolución

    ejes[i].set_ylabel("Error")  # Etiqueta del eje Y
    ejes[i].set_xlim(tiempo_inicio, tiempo_fin)  # Límites de X: el fragmento
    ejes[i].grid(True, alpha = 0.3)  # Cuadrícula tenue

ejes[-1].set_xlabel("Tiempo (segundos)")  # Etiqueta de X en la última gráfica

plt.suptitle(  # Título general
    "Comparación del error de cuantificación",  # Texto del título
    fontsize = 15)  # Tamaño de letra

plt.tight_layout()  # Ajusta márgenes
plt.show()  # Muestra la figura

# 11. Reproducción de señales cuantificadas

In [ ]:
print("=" * 55)  # Línea separadora
print("SEÑAL ORIGINAL")  # Título de la sección
print(f"Frecuencia de muestreo: {fs} Hz")  # Frecuencia de muestreo
print(f"Número de muestras: {len(senal)}")  # Total de muestras

display(Audio(senal, rate = fs, normalize = False))  # Reproduce la señal original

for bits in resoluciones:  # Recorre cada resolución

    senal_actual = senales_cuantificadas[bits]  # Señal cuantificada con esos bits

    niveles = 2 ** bits  # Total de niveles posibles
    sqnr = resultados_sqnr[bits]  # SQNR calculada en la celda 10
    rms = np.sqrt(np.mean(senal_actual**2))  # RMS de la señal cuantificada

    print("\n" + "=" * 55)  # Línea en blanco y separador
    print(f"CUANTIFICACIÓN DE {bits} BITS")  # Resolución actual
    print(f"Niveles posibles: {niveles}")  # Niveles posibles
    print(f"Frecuencia de muestreo: {fs} Hz")  # Frecuencia de muestreo
    print(f"Amplitud RMS: {rms:.6f}")  # RMS con 6 decimales
    print(f"SQNR: {sqnr:.2f} dB")  # SQNR con 2 decimales

    display(Audio(senal_actual, rate = fs, normalize = False))  # Reproduce la señal cuantificada